In [1]:
MODE = "test"                # "test" (teammate C) or "train" (teammate D)
SMOKE = False                # True: quick code check (~3-5 min, interactive)
                             # False: full run (use Save Version -> Save & Run All)
SMOKE_QUERIES = 2_000        # smoke: queries per country
SMOKE_POOL = 200_000         # smoke: pool records per country (a slice: recall is NOT meaningful)
 
DATA_ROOT = "/kaggle/input/datasets/mahimasunilbachhav/amazon-ml-challenge/student_resource/dataset"
INPUT_ROOT = "/kaggle/input"
WORK = "/kaggle/working"
 
K5, K7 = 50, 50              # neighbours per query for L5 and L7
L6_CAP, L6_MAX = 50, 50      # drop exact-key buckets larger than L6_CAP; keep <= L6_MAX per query
MIN_SIM = 0.10
DIM_ADDR, DIM_SKEL = 256, 128
TFIDF_FIT_ROWS, SVD_FIT_ROWS, TFIDF_CHUNK = 1_000_000, 300_000, 250_000
Q_BATCH = 1024
FEAT_CHUNK = 2_000_000
SEED = 42

In [2]:
import gc
import hashlib
import json
import shutil
import subprocess
import sys
import time
import zipfile
from collections import Counter, defaultdict
from pathlib import Path
 
import numpy as np
import pandas as pd
import psutil
import lightgbm as lgb
from tqdm.auto import tqdm
 
try:
    from rapidfuzz.process import cpdist
except ImportError:
    print("installing/upgrading rapidfuzz (needs Internet ON) ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "rapidfuzz"], check=True)
    try:
        from rapidfuzz.process import cpdist
    except ImportError:
        raise SystemExit("rapidfuzz was upgraded: restart the session, then rerun this cell.")
from rapidfuzz import fuzz
from rapidfuzz.distance import JaroWinkler
 
DATA_ROOT, WORK, INPUT_ROOT = Path(DATA_ROOT), Path(WORK), Path(INPUT_ROOT)
T_START = time.time()
TIMES = {}                     # stage -> [seconds, scalable?]
 
 
def log(msg):
    print(f"  [{(time.time()-T_START)/60:5.1f} min] {msg}", flush=True)
 
 
def mem(tag=""):
    rss = psutil.Process().memory_info().rss / 1e9
    print(f"    [MEM] {tag:<28} RAM={rss:5.1f} GB  free={psutil.virtual_memory().available/1e9:5.1f} GB",
          flush=True)
 
 
def bar(it=None, **kw):
    kw.setdefault("mininterval", 10)
    return tqdm(it, **kw)
 
 
def timed(stage, t0, scalable=True):
    """Accumulate wall time per stage. scalable: True = grows with the number of
    pairs, False = fixed cost, None = done once and reused by the full run."""
    TIMES.setdefault(stage, [0.0, scalable])[0] += time.time() - t0
 
 
def hash_bucket(eid):
    """Deterministic 0-9999 bucket; identical to the EDA / Notebook 2 split."""
    return int(hashlib.md5(eid.encode()).hexdigest(), 16) % 10000
 
 
def locate(marker):
    """All files named `marker` under INPUT_ROOT, plus those inside any .zip under
    INPUT_ROOT. Each zip holding the marker is unpacked once into
    WORK/_unzipped/<zip name>/ (Kaggle may keep uploaded zips packed)."""
    root_ok = INPUT_ROOT.exists()
    hits = sorted(INPUT_ROOT.rglob(marker)) if root_ok else []
    for z in (sorted(INPUT_ROOT.rglob("*.zip")) if root_ok else []):
        dest = WORK / "_unzipped" / z.stem
        done = dest / ".unpacked"
        if not done.exists():
            with zipfile.ZipFile(z) as zf:
                if not any(n.endswith("/" + marker) or n == marker for n in zf.namelist()):
                    continue
                log(f"unpacking {z.name} -> {dest}")
                zf.extractall(dest)
            done.touch()
        hits += sorted(dest.rglob(marker))
    return hits
 
 
def project(full_factor, label):
    """Print measured stage times and their projection to the full run."""
    print(f"\n  RUNTIME PROJECTION ({label})")
    print(f"    {'stage':<30}{'smoke (s)':>11}{'full (min)':>12}")
    total = 0.0
    for stage, (sec, scalable) in TIMES.items():
        # scalable None = already done and reused by the full run
        full = 0.0 if scalable is None else sec * (full_factor if scalable else 1.0)
        total += full
        print(f"    {stage:<30}{sec:>11.1f}{full/60:>12.1f}")
    print(f"    {'TOTAL':<30}{'':>11}{total/60:>12.1f}")
    return total

installing/upgrading rapidfuzz (needs Internet ON) ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 30.1 MB/s eta 0:00:00


In [3]:
COLS = ["entity_id", "business_name", "business_address", "country"]
 
 
def load_src(path):
    """Read a source TSV. sep='\t' is mandatory: fields contain commas."""
    return pd.read_csv(path, sep="\t", dtype=str, names=COLS, header=0, keep_default_na=False)
 
 
# ---- cleaning: copied verbatim from Notebook 2 so both stages agree ----------
LEGAL_RE = (r"\b(llc|l\.l\.c|inc|corp|corporation|ltd|limited|pvt|private|"
            r"llp|lp|pllc|sas|sarl|eurl|sasu|snc|company)\b")
 
ADDR_ABBR = [
    (r"\brd\b", "road"), (r"\bst\b", "street"), (r"\bdr\b", "drive"),
    (r"\bave\b", "avenue"), (r"\bblvd\b", "boulevard"), (r"\bln\b", "lane"),
    (r"\bct\b", "court"), (r"\bcir\b", "circle"), (r"\bhno\b", "house number"),
    (r"\bno\b", "number"), (r"\bopp\b", "opposite"), (r"\bnr\b", "near"),
    (r"\bdist\b", "district"), (r"\bflr\b", "floor"),
]
 
 
def clean_text(s):
    """Shared cleaning. Non-Latin scripts are dropped here; L3 reads raw text."""
    s = s.fillna("").astype(str).str.lower()
    s = (s.str.normalize("NFKD")
          .str.encode("ascii", errors="ignore")
          .str.decode("ascii"))
    s = s.str.replace(r"##\s*", "", regex=True)
    s = s.str.replace(r"\bwww\.", "", regex=True)
    s = s.str.replace(r"\.(com|net|org|co|in|fr|uk|biz|info|io)\b", "", regex=True)
    s = s.str.replace(r"^[\s\-\.\,\!\@\#\$\%\^\&\*\>\<\'\"/\\]+", "", regex=True)
    s = s.str.replace("&", " and ", regex=False)
    s = s.str.replace(r"\bn/?a\b", " ", regex=True)
    s = s.str.replace(r"[^\w\s]", " ", regex=True)
    return s.str.replace(r"\s+", " ", regex=True).str.strip()
 
 
def clean_name(s):
    """Name: shared cleaning + legal suffixes + repeated words (FH FH -> FH)."""
    s = clean_text(s)
    s = s.str.replace(LEGAL_RE, " ", regex=True)
    s = s.str.replace(r"\b(\w+)\s+\1\b", r"\1", regex=True)
    return s.str.replace(r"\s+", " ", regex=True).str.strip()
 
 
def clean_addr(s):
    """Address: shared cleaning + abbreviation expansion."""
    s = clean_text(s)
    for pat, rep in ADDR_ABBR:
        s = s.str.replace(pat, rep, regex=True)
    return s.str.replace(r"\s+", " ", regex=True).str.strip()
 
 
# ---- extra text columns for Stage 2 -------------------------------------------
# Street-type words and French articles are skipped when forming the address key,
# so "15 Rue de la Hoxie" -> "15_hoxie" (not "15_rue"), matching "351 Hoxie Avenue"
# style keys. This matters most for France, which training never sees.
SKIP_AFTER_NUMBER = {
    "road", "street", "drive", "avenue", "boulevard", "lane", "court", "circle",
    "way", "highway", "rue", "impasse", "allee", "chemin", "place", "route", "quai",
    "cours", "square", "de", "du", "des", "la", "le", "les", "l", "d",
}
 
 
def addr_key_string(caddr):
    """House number + the first following token that isn't a street word/article.
    Returned as a space-joined, sorted string so rapidfuzz can compare two of them."""
    toks = caddr.split()
    keys = set()
    for i, a in enumerate(toks):
        if a.isdigit():
            a2 = a.lstrip("0")
            j = i + 1
            while j < len(toks) and toks[j] in SKIP_AFTER_NUMBER:
                j += 1
            if a2 and j < len(toks):
                b = toks[j].lstrip("0") if toks[j].isdigit() else toks[j]
                if b:
                    keys.add(a2 + "_" + b)
    return " ".join(sorted(keys))
 
 
def prepare_texts(df):
    """All per-record text columns Stage 2 needs, computed once."""
    out = pd.DataFrame({"entity_id": df["entity_id"].values})
    out["cname"] = clean_name(df["business_name"]).values
    out["caddr"] = clean_addr(df["business_address"]).values
    out["digits"] = out["caddr"].str.findall(r"\b\d+\b").str.join(" ")
    out["nospace"] = out["cname"].str.replace(" ", "", regex=False)
    out["akeys"] = [addr_key_string(a) for a in out["caddr"]]
    out["nonlatin"] = np.array([any(ord(ch) > 0x024F for ch in s)
                                for s in df["business_name"]], dtype=np.int8)
    return out

In [4]:
def group_features(q, S):
    """Scores relative to each S1's own candidate list (q must be sorted).
    Scale-free features like rank and gap-to-best transfer across countries."""
    starts = np.flatnonzero(np.r_[True, q[1:] != q[:-1]])
    sizes = np.diff(np.r_[starts, len(q)])
    cols = []
    for j in range(S.shape[1]):
        s = S[:, j].astype(np.float32)
        gmax = np.repeat(np.maximum.reduceat(s, starts), sizes)
        order = np.lexsort((-s, q))
        rank = np.empty(len(q), np.float32)
        rank[order] = np.arange(len(q)) - np.repeat(starts, sizes)
        cols += [s, gmax - s, rank]
    cols += [(S[:, :3] > 0).sum(1) + (S[:, 3] > 0), np.repeat(sizes, sizes)]
    return np.column_stack(cols).astype(np.float32)
 
 
SCORERS = [("cname", "ratio", fuzz.ratio), ("cname", "token_set", fuzz.token_set_ratio),
           ("cname", "token_sort", fuzz.token_sort_ratio), ("cname", "partial", fuzz.partial_ratio),
           ("cname", "jaro_winkler", JaroWinkler.normalized_similarity),
           ("nospace", "ratio", fuzz.ratio), ("caddr", "token_set", fuzz.token_set_ratio),
           ("caddr", "token_sort", fuzz.token_sort_ratio), ("caddr", "ratio", fuzz.ratio),
           ("digits", "token_set", fuzz.token_set_ratio), ("akeys", "token_set", fuzz.token_set_ratio)]
FLAG_FEATURES = ["a_blank", "q_blank", "p_blank", "nonlatin", "len_diff"]
PRE_FEATURES = [f"{s}_{k}" for s in ("l1", "l2", "l3", "ak") for k in ("score", "gap", "rank")] \
    + ["n_layers", "n_cands", "is_s3"]
RICH_FEATURES = PRE_FEATURES + ["pf", "pf_gap", "pf_rank"] \
    + [f"{col}_{nm}" for col, nm, _ in SCORERS] + FLAG_FEATURES
 
 
def string_features(qt, pt, q, p):
    """String similarities + flags for pairs (q[i], p[i]); qt/pt are the prepared
    text tables aligned to qids/pids. Chunked to bound memory."""
    Q = {c: qt[c].to_numpy() for c in ("cname", "caddr", "digits", "nospace", "akeys")}
    P = {c: pt[c].to_numpy() for c in ("cname", "caddr", "digits", "nospace", "akeys")}
    qlen, plen = qt["cname"].str.len().to_numpy(), pt["cname"].str.len().to_numpy()
    q_nl, p_nl = qt["nonlatin"].to_numpy(), pt["nonlatin"].to_numpy()
    out = np.empty((len(q), len(SCORERS) + len(FLAG_FEATURES)), dtype=np.float32)
    for i in bar(range(0, len(q), FEAT_CHUNK), desc="string features", unit="chunk"):
        qs, ps = q[i:i + FEAT_CHUNK], p[i:i + FEAT_CHUNK]
        n = len(qs)
        for k, (col, _, sc) in enumerate(SCORERS):
            out[i:i + n, k] = cpdist(Q[col][qs].tolist(), P[col][ps].tolist(),
                                     scorer=sc, workers=-1)
        j = len(SCORERS)
        out[i:i + n, j] = (Q["caddr"][qs] == "") | (P["caddr"][ps] == "")
        out[i:i + n, j + 1] = Q["cname"][qs] == ""
        out[i:i + n, j + 2] = P["cname"][ps] == ""
        out[i:i + n, j + 3] = q_nl[qs] | p_nl[ps]
        out[i:i + n, j + 4] = np.abs(qlen[qs] - plen[ps])
    return out
 
 
 
 
# ---- v2: transliteration, legal form, phonetic skeleton (shared) -------------
import re as _re
import unicodedata as _ud
try:
    from indic_transliteration import sanscript as _sanscript
    from indic_transliteration.sanscript import transliterate as _translit
except ImportError:
    print("installing indic_transliteration (MIT licence; needs Internet ON) ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "indic_transliteration"], check=True)
    from indic_transliteration import sanscript as _sanscript
    from indic_transliteration.sanscript import transliterate as _translit
 
_BLOCK_SCRIPT = {0x0900: _sanscript.DEVANAGARI, 0x0980: _sanscript.BENGALI,
                 0x0A00: _sanscript.GURMUKHI, 0x0A80: _sanscript.GUJARATI,
                 0x0B00: _sanscript.ORIYA, 0x0B80: _sanscript.TAMIL,
                 0x0C00: _sanscript.TELUGU, 0x0C80: _sanscript.KANNADA,
                 0x0D00: _sanscript.MALAYALAM}
_LEGAL_TOKENS = {"pvt", "private", "pr", "pra", "ltd", "limited", "li", "llp", "pllc", "llc",
                 "lp", "inc", "incorporated", "corp", "corporation", "co", "company", "sas",
                 "sasu", "sarl", "eurl", "snc", "sa", "public"}
_LEGAL_PREFIX = ("prai", "pirai", "limi", "elel")    # transliterated private / limited / LLP
_FORM_KEYS = (("LLP", {"llp"}), ("PLLC", {"pllc"}), ("LLC", {"llc"}), ("LP", {"lp"}),
              ("INC", {"inc", "incorporated"}), ("CORP", {"corp", "corporation"}),
              ("SASU", {"sasu"}), ("SAS", {"sas"}), ("SARL", {"sarl"}), ("EURL", {"eurl"}),
              ("SNC", {"snc"}), ("SA", {"sa"}))
_SKEL_FOLD = (("ph", "f"), ("th", "t"), ("sh", "s"), ("kh", "k"), ("gh", "g"), ("bh", "b"),
              ("dh", "d"), ("ch", "c"), ("ck", "k"), ("c", "k"), ("q", "k"), ("x", "ks"),
              ("z", "s"), ("w", "v"), ("j", "g"), ("y", ""), ("v", "b"))   # Bengali renders B as v
 
 
def latin_tokens(name):
    """Lowercase Latin tokens of a name. Indic-script words are transliterated
    (ITRANS), accents removed, dots dropped (L.L.C -> llc), other punctuation
    splits words (Housing-Network -> housing network)."""
    s = str(name)
    s = _re.sub(r"\.(com|net|org|co|in|fr|uk|biz|info|io)\b", " ", s, flags=_re.I)
    s = s.replace(".", "")
    fast = all(ord(ch) <= 0x024F for ch in s)
    if fast:
        s = _re.sub(r"[^\w\s]", " ", s)
    else:
        # Indic vowel signs / virama are not \w, so only replace real punctuation
        # and symbols (Unicode categories P* and S*).
        s = "".join(" " if _ud.category(ch)[0] in "PS" else ch for ch in s)
    out = []
    for w in s.split():
        indic = False
        if not fast:
            blk = next((b for b in _BLOCK_SCRIPT if any(b <= ord(c) < b + 0x80 for c in w)), None)
            if blk is not None:
                indic = True
                try:
                    w = _translit(w, _BLOCK_SCRIPT[blk], _sanscript.ITRANS)
                except Exception:
                    pass
                if blk == 0x0B80:                     # Tamil: library leaves some letters
                    w = w.replace("\u0BA9", "n").replace("\u0BB1", "r")
        w = _ud.normalize("NFKD", w).encode("ascii", "ignore").decode().lower()
        if indic and blk == 0x0B80:
            # Tamil has no aspirated/voiced contrast; the library writes ச க ப ட/த
            # as jh gh bh dh — fold them to s k p t.
            for a, b in (("jh", "s"), ("gh", "k"), ("bh", "p"), ("dh", "t")):
                w = w.replace(a, b)
        w = _re.sub(r"[^a-z0-9]", "", w)
        if indic:
            w = _re.sub(r"([bcdfghjklmnpqrstvwxyz])a$", r"\1", w)    # inherent final 'a'
        if w:
            out.append(w)
    return out
 
 
def legal_form(toks):
    """Canonical legal form: PVT (private limited), PLC, LTD, LLP, LLC, INC, ...
    Spelling variants map to one code (Pvt Ltd = Private Limited = प्रा. लि.)."""
    s = set(toks)
    has_pvt = bool(s & {"pvt", "private", "pr", "pra"}) or any(t.startswith(("prai", "pirai")) for t in toks)
    has_ltd = bool(s & {"ltd", "limited", "li"}) or any(t.startswith("limi") for t in toks)
    if has_pvt:
        return "PVT"
    if "public" in s and has_ltd:
        return "PLC"
    if any(t.startswith("elel") for t in toks):
        return "LLP"
    for code, keys in _FORM_KEYS:
        if s & keys:
            return code
    if has_ltd:
        return "LTD"
    if s & {"co", "company"}:
        return "CO"
    return ""
 
 
def skeleton(toks):
    """Phonetic consonant skeleton of the non-legal tokens: folds spelling variants,
    drops vowels, collapses repeats. 'teknolojis' and 'technologies' -> 'tknlgs'."""
    words = [t for t in toks if t not in _LEGAL_TOKENS and not t.startswith(_LEGAL_PREFIX)]
    s = " ".join(words)
    for a, b in _SKEL_FOLD:
        s = s.replace(a, b)
    s = _re.sub(r"[aeiou]", "", s)
    s = _re.sub(r"(.)\1+", r"\1", s).replace("m", "n")
    return _re.sub(r"\s+", " ", s).strip()
 
 
def prepare_extra(names):
    """Per-record legal form and phonetic skeleton."""
    forms, skels = [], []
    for nm in names:
        toks = latin_tokens(nm)
        forms.append(legal_form(toks))
        skels.append(skeleton(toks))
    return pd.DataFrame({"form": forms, "skel": skels})
 
 
EXTRA_FEATURES = ["form_same", "form_diff", "form_one_missing", "name_exact",
                  "skel_ratio", "skel_tset"]
RICH_V2 = RICH_FEATURES + EXTRA_FEATURES          # v1 features first, then v2 additions
 
 
def string_features_v2(qt, pt, q, p):
    """v1 string features + legal form relation, exact name, skeleton similarity.
    qt/pt must include the v2 columns 'form' and 'skel'."""
    base = string_features(qt, pt, q, p)
    fq, fp = qt["form"].to_numpy()[q], pt["form"].to_numpy()[p]
    cq, cp = qt["cname"].to_numpy()[q], pt["cname"].to_numpy()[p]
    sq, sp = qt["skel"].to_numpy()[q].tolist(), pt["skel"].to_numpy()[p].tolist()
    both = (fq != "") & (fp != "")
    extra = np.column_stack([
        both & (fq == fp),
        both & (fq != fp),
        (fq == "") != (fp == ""),
        (cq != "") & (cq == cp),
        cpdist(sq, sp, scorer=fuzz.ratio, workers=-1),
        cpdist(sq, sp, scorer=fuzz.token_set_ratio, workers=-1),
    ]).astype(np.float32)
    return np.column_stack([base, extra]).astype(np.float32)
 
 
def ensure_extra(prep_dir, ids_path, side, raw_names_by_id):
    """Cache <side>_x.parquet (form, skel) aligned to ids; compute if missing."""
    path = prep_dir / f"{side}_x.parquet"
    if path.exists():
        return
    ids = np.load(ids_path, allow_pickle=True)
    names = raw_names_by_id.reindex(ids)
    assert names.notna().all(), f"{path}: ids missing from the source files"
    t = time.time()
    x = prepare_extra(names.tolist())
    x.insert(0, "entity_id", ids)
    x.to_parquet(path, index=False)
    log(f"{path.parent.name}/{path.name}: {len(ids):,} rows in {time.time()-t:.0f}s")
 
 
def load_prepared(prep_dir, side):
    """v1 prepared text table + v2 extra columns, row-aligned."""
    t = pd.read_parquet(prep_dir / f"{side}.parquet")
    x = pd.read_parquet(prep_dir / f"{side}_x.parquet")
    assert (t["entity_id"].values == x["entity_id"].values).all(), "extra columns misaligned"
    t["form"], t["skel"] = x["form"].values, x["skel"].values
    return t

installing indic_transliteration (MIT licence; needs Internet ON) ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.9/162.9 kB 3.4 MB/s eta 0:00:00


In [5]:
import os
import pickle
import torch
from concurrent.futures import ProcessPoolExecutor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
 
assert MODE in ("test", "train")
assert torch.cuda.is_available(), "GPU not enabled: set Accelerator to GPU T4"
DEV = "cuda"
 
 
def purge():
    gc.collect()
    torch.cuda.empty_cache()
 
 
# ---- GPU exact top-k (same algorithm as Notebook 2) ---------------------------
def gpu_topk(Q, P, k, desc, q_batch=Q_BATCH, p_chunk=1_000_000):
    """Exact inner-product top-k on the GPU (fp16); -1 where score < MIN_SIM."""
    k = min(k, P.shape[0])
    P_t = torch.from_numpy(np.ascontiguousarray(P, dtype=np.float16)).to(DEV)
    I = np.empty((Q.shape[0], k), dtype=np.int32)
    S = np.empty((Q.shape[0], k), dtype=np.float16)
    for i in bar(range(0, Q.shape[0], q_batch), desc=desc, unit="batch"):
        q = torch.from_numpy(np.ascontiguousarray(Q[i:i + q_batch], dtype=np.float16)).to(DEV)
        best_v = best_i = None
        for j in range(0, P_t.shape[0], p_chunk):
            s = q @ P_t[j:j + p_chunk].T
            v, ix = torch.topk(s, min(k, s.shape[1]), dim=1)
            ix += j
            if best_v is None:
                best_v, best_i = v, ix
            else:
                cv, ci = torch.cat([best_v, v], dim=1), torch.cat([best_i, ix], dim=1)
                best_v, sel = torch.topk(cv, k, dim=1)
                best_i = torch.gather(ci, 1, sel)
            del s
        n = q.shape[0]
        I[i:i + n] = best_i.cpu().numpy()
        S[i:i + n] = best_v.cpu().numpy()
    del P_t
    purge()
    I[S < MIN_SIM] = -1
    return I, S
 
 
def fit_tfidf_svd(p_txt, dim, ngram):
    """Char n-gram TF-IDF (vocabulary from a sample) + SVD fitted on a sample."""
    rng = np.random.default_rng(SEED)
    idx = rng.choice(len(p_txt), min(TFIDF_FIT_ROWS, len(p_txt)), replace=False)
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=ngram, min_df=3, max_df=0.5,
                          max_features=40_000, sublinear_tf=True, dtype=np.float32)
    X = vec.fit_transform([p_txt[i] for i in idx])
    dim = min(dim, X.shape[1] - 1)
    sidx = rng.choice(X.shape[0], min(SVD_FIT_ROWS, X.shape[0]), replace=False)
    svd = TruncatedSVD(n_components=dim, algorithm="randomized", n_iter=4,
                       random_state=SEED).fit(X[sidx])
    return vec, svd
 
 
def tfidf_transform(vec, svd, texts, desc):
    out = np.empty((len(texts), svd.n_components), dtype=np.float16)
    for i in bar(range(0, len(texts), TFIDF_CHUNK), desc=desc, unit="chunk"):
        out[i:i + TFIDF_CHUNK] = normalize(svd.transform(vec.transform(texts[i:i + TFIDF_CHUNK])))
    return out
 
 
def _skel_chunk(names):
    return [skeleton(latin_tokens(n)) for n in names]
 
 
def skeletons(names, chunk=100_000):
    """Phonetic skeletons in parallel over all CPU cores."""
    parts = [names[i:i + chunk] for i in range(0, len(names), chunk)]
    out = []
    with ProcessPoolExecutor(max_workers=os.cpu_count() or 2) as ex:
        for r in bar(ex.map(_skel_chunk, parts), total=len(parts), desc="skeletons", unit="chunk"):
            out.extend(r)
    return out
 
 
def exact_key_layer(q_keys, p_keys):
    """L6: pool rows sharing an exact key; score = number of keys shared."""
    buckets = defaultdict(list)
    for j, ks in enumerate(bar(p_keys, desc="L6 index", unit="row", unit_scale=True)):
        for key in ks:
            buckets[key].append(j)
    for key in [k for k, v in buckets.items() if len(v) > L6_CAP]:
        del buckets[key]
    I = np.full((len(q_keys), L6_MAX), -1, dtype=np.int32)
    V = np.zeros((len(q_keys), L6_MAX), dtype=np.float16)
    for i, ks in enumerate(q_keys):
        cnt = Counter()
        for key in ks:
            b = buckets.get(key)
            if b:
                cnt.update(b)
        if cnt:
            best = sorted(cnt.items(), key=lambda kv: (-kv[1], kv[0]))[:L6_MAX]
            I[i, :len(best)] = [b[0] for b in best]
            V[i, :len(best)] = [b[1] for b in best]
    return I, V
 
 
def keys_of(cname, nospace, skel):
    """Exact-match keys, prefixed by type; short keys are too ambiguous to use."""
    return [tuple(k for k in (("n:" + a) if len(a) >= 3 else None,
                              ("s:" + b) if len(b) >= 4 else None,
                              ("k:" + c) if len(c) >= 3 else None) if k)
            for a, b, c in zip(cname, nospace, skel)]
 
 
def merge_layers(layers, nq, n_pool):
    """Union of layers -> flat arrays sorted by (q, p); one score column per layer."""
    L = len(layers)
    parts = defaultdict(list)
    for r0 in range(0, nq, 50_000):
        r1 = min(nq, r0 + 50_000)
        bi = [I[r0:r1] for I, _ in layers]
        bv = [V[r0:r1].astype(np.float32) for _, V in layers]
        widths = [b.shape[1] for b in bi]
        Pm, Vm = np.concatenate(bi, axis=1), np.concatenate(bv, axis=1)
        Lm = np.broadcast_to(np.repeat(np.arange(L), widths), Pm.shape)
        Qm = np.broadcast_to(np.arange(r0, r1, dtype=np.int64)[:, None], Pm.shape)
        ok = Pm >= 0
        keys = Qm[ok] * n_pool + Pm[ok].astype(np.int64)
        uk, inv = np.unique(keys, return_inverse=True)
        feats = np.zeros((L, len(uk)), dtype=np.float32)
        feats[Lm[ok], inv] = Vm[ok]
        parts["q"].append((uk // n_pool).astype(np.int32))
        parts["p"].append((uk % n_pool).astype(np.int32))
        for li in range(L):
            parts[f"s{li}"].append(feats[li].astype(np.float16))
    return {k: (np.concatenate(v) if v else np.zeros(0)) for k, v in parts.items()}

In [6]:
CDIR = {}
for f in locate("pairs.npz"):
    if f.parent.parent.name == f"ckpt_{MODE}" and (f.parent / "qids.npy").exists():
        CDIR[f.parent.name] = f.parent
assert CDIR, f"ckpt_{MODE} not found: attach the {MODE} blocking output (zips or folders)"
COUNTRIES = sorted(CDIR)
OUT = WORK / f"ckpt_b_{MODE}{'_smoke' if SMOKE else ''}"
# Working cache lives in /tmp: it is not saved as notebook output (a committed run
# would otherwise save ~7 GB of intermediate vectors) and does not count against
# the 20 GB /kaggle/working limit. Smoke and full runs never share a cache.
CACHE = Path("/tmp") / f"_cache_b_{MODE}{'_smoke' if SMOKE else ''}"
OUT.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)
print("=" * 70)
print(f"PHASE B  MODE={MODE}  {'SMOKE RUN' if SMOKE else 'FULL RUN'}  countries={COUNTRIES}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print("=" * 70)
 
GT = {}
if MODE == "train":
    gt = pd.read_csv(DATA_ROOT / "train" / "train_ground_truth.tsv", sep="\t", dtype=str,
                     names=["s1", "m"], header=0, keep_default_na=False)
    GT = {s: {x for x in m.split(",") if x} for s, m in zip(gt["s1"], gt["m"])}
    del gt

PHASE B  MODE=test  FULL RUN  countries=['France', 'India', 'US']
GPU: Tesla T4


In [7]:
t0 = time.time()
todo = [c for c in COUNTRIES if not (CACHE / c / "texts.done").exists()]
if todo:
    recs = pd.concat([load_src(DATA_ROOT / MODE / f"{MODE}_source{i}.tsv") for i in (1, 2, 3)],
                     ignore_index=True)
    row_of = pd.Index(recs["entity_id"])
    log(f"loaded {len(recs):,} {MODE} records")
    for c in todo:
        (CACHE / c).mkdir(parents=True, exist_ok=True)
        for side in ("q", "p"):
            ids = np.load(CDIR[c] / f"{side}ids.npy", allow_pickle=True)
            if SMOKE:
                ids = ids[:SMOKE_QUERIES] if side == "q" else ids[:SMOKE_POOL]
            rows = row_of.get_indexer(ids)
            assert (rows >= 0).all(), f"{c}/{side}: ids missing from the {MODE} files"
            sub = recs.iloc[rows]
            t = time.time()
            df = pd.DataFrame({"cname": clean_name(sub["business_name"]).values,
                               "caddr": clean_addr(sub["business_address"]).values})
            df["nospace"] = df["cname"].str.replace(" ", "", regex=False)
            df["skel"] = skeletons(sub["business_name"].tolist())
            df.to_parquet(CACHE / c / f"{side}_text.parquet", index=False)
            log(f"[{c}] {side}: {len(df):,} rows in {time.time()-t:.0f}s")
        (CACHE / c / "texts.done").touch()
    del recs, row_of
    gc.collect()
timed("text columns", t0, scalable="pool")

  [  0.7 min] loaded 11,702,133 test records


skeletons:   0%|          | 0/3 [00:00<?, ?chunk/s]

  [  1.1 min] [France] q: 259,452 rows in 13s


skeletons:   0%|          | 0/15 [00:01<?, ?chunk/s]

  [  2.2 min] [France] p: 1,434,993 rows in 67s


...

  [  8.5 min] [India] p: 4,717,565 rows in 319s


skeletons:   0%|          | 0/7 [00:00<?, ?chunk/s]

  [  9.0 min] [US] q: 663,106 rows in 30s


skeletons:   0%|          | 0/39 [00:00<?, ?chunk/s]

  [ 11.9 min] [US] p: 3,817,031 rows in 169s


In [8]:
full_q = used_q = full_p = used_p = 0
for c in COUNTRIES:
    qids = np.load(CDIR[c] / "qids.npy", allow_pickle=True)
    pids = np.load(CDIR[c] / "pids.npy", allow_pickle=True)
    nq = min(SMOKE_QUERIES, len(qids)) if SMOKE else len(qids)
    full_q += len(qids)
    used_q += nq
    full_p += len(pids)
    used_p += min(SMOKE_POOL, len(pids)) if SMOKE else len(pids)
    if SMOKE:
        pids = pids[:SMOKE_POOL]
    if (OUT / c / "pairs_b.npz").exists():
        log(f"[{c}] already done — skipping")
        continue
    (OUT / c).mkdir(parents=True, exist_ok=True)
    qt = pd.read_parquet(CACHE / c / "q_text.parquet").iloc[:nq]
    pt = pd.read_parquet(CACHE / c / "p_text.parquet")
    log(f"[{c}] {nq:,} queries vs {len(pt):,} pool records")
    layers = []
    for name, col, dim, ngram, k in (("L5", "caddr", DIM_ADDR, (3, 4), K5),
                                     ("L7", "skel", DIM_SKEL, (2, 3), K7)):
        t0 = time.time()
        mpath, ppath = CACHE / c / f"{name}_model.pkl", CACHE / c / f"{name}_pool.npy"
        if ppath.exists():
            vec, svd = pickle.load(open(mpath, "rb"))
            P = np.load(ppath)
        else:
            vec, svd = fit_tfidf_svd(pt[col].tolist(), dim, ngram)
            P = tfidf_transform(vec, svd, pt[col].tolist(), f"{name} pool")
            pickle.dump((vec, svd), open(mpath, "wb"))
            np.save(ppath, P)
        timed(f"{name} pool vectors", t0, scalable="pool")
        t0 = time.time()
        Q = tfidf_transform(vec, svd, qt[col].tolist(), f"{name} queries")
        layers.append(gpu_topk(Q, P, k, f"{name} search {c}"))
        del P, Q, vec, svd
        purge()
        timed(f"{name} queries + GPU search", t0, scalable="both")
    t0 = time.time()
    I6, V6 = exact_key_layer(keys_of(qt["cname"], qt["nospace"], qt["skel"]),
                             keys_of(pt["cname"], pt["nospace"], pt["skel"]))
    timed("L6 exact keys", t0, scalable="pool")
    t0 = time.time()
    L5, L7 = layers
    m = merge_layers([L5, (I6, V6), L7], nq, len(pt))
    out = {"q": m["q"], "p": m["p"], "l5": m["s0"], "l6": m["s1"], "l7": m["s2"]}
    stats = {"queries": nq, "pool": int(len(pt)), "pairs": int(len(out["q"])),
             "pairs_per_query": float(len(out["q"]) / max(nq, 1))}
 
    if MODE == "train":
        # label the new pairs; recall of old / new / combined candidates
        ppos = {pid: j for j, pid in enumerate(pids)}
        true_keys = []
        n_true = np.zeros(nq, dtype=np.int64)
        for i in range(nq):
            ms = GT.get(qids[i], ())
            n_true[i] = len(ms)
            for mid in ms:
                j = ppos.get(mid)
                if j is not None:
                    true_keys.append(i * len(pids) + j)
        true_keys = np.unique(np.array(true_keys, dtype=np.int64))
        new_keys = out["q"].astype(np.int64) * len(pids) + out["p"]
        out["label"] = np.isin(new_keys, true_keys).astype(np.int8)
        d_old = np.load(CDIR[c] / "pairs.npz")
        oq = d_old["q"]
        cut = int(np.searchsorted(oq, nq))
        old_keys = oq[:cut].astype(np.int64) * len(pids) + d_old["p"][:cut]
        qb = np.array([hash_bucket(x) for x in qids[:nq]], dtype=np.int16)
 
        def recall(keys, sel_q):
            kk = keys[np.isin(keys // len(pids), np.flatnonzero(sel_q))]
            found = np.isin(true_keys[np.isin(true_keys // len(pids), np.flatnonzero(sel_q))], kk)
            return int(found.sum()) / max(int(n_true[sel_q].sum()), 1)
 
        allq, heldq = np.ones(nq, bool), qb < 500
        comb = np.union1d(old_keys, new_keys)
        stats.update({
            "recall_old": recall(old_keys, allq), "recall_new": recall(new_keys, allq),
            "recall_combined": recall(comb, allq),
            "recall_old_heldout": recall(old_keys, heldq), "recall_combined_heldout": recall(comb, heldq),
            "recall_by_layer": {nm: recall(new_keys[out[col] > 0], allq)
                                for nm, col in (("L5 address", "l5"), ("L6 exact keys", "l6"),
                                                ("L7 skeleton", "l7"))},
            "new_pairs_not_in_old": int((~np.isin(new_keys, old_keys)).sum()),
        })
        del d_old, old_keys, comb
    np.savez(OUT / c / "pairs_b.npz", **out)
    json.dump(stats, open(OUT / c / "stats_b.json", "w"), indent=2)
    timed("merge + save", t0, scalable="query")
    msg = f"[{c}] {stats['pairs']:,} pairs ({stats['pairs_per_query']:.0f}/query)"
    if MODE == "train":
        msg += (f"   recall old {stats['recall_old']*100:.2f}% -> combined "
                f"{stats['recall_combined']*100:.2f}%")
    log(msg)
    del qt, pt, layers, I6, V6, out, m
    purge()
    mem(f"after {c}")

  [ 12.0 min] [France] 259,452 queries vs 1,434,993 pool records


L5 pool:   0%|          | 0/6 [00:00<?, ?chunk/s]

L5 queries:   0%|          | 0/2 [00:00<?, ?chunk/s]

L5 search France:   0%|          | 0/254 [00:00<?, ?batch/s]

L7 pool:   0%|          | 0/6 [00:00<?, ?chunk/s]

...

L7 pool:   0%|          | 0/16 [00:00<?, ?chunk/s]

L7 queries:   0%|          | 0/3 [00:00<?, ?chunk/s]

L7 search US:   0%|          | 0/648 [00:00<?, ?batch/s]

L6 index:   0%|          | 0.00/3.82M [00:00<?, ?row/s]

  [ 55.4 min] [US] 65,322,744 pairs (99/query)
    [MEM] after US                     RAM=  8.2 GB  free= 23.8 GB


In [9]:
print("\n" + "=" * 70)
print(f"PHASE B {MODE.upper()} {'SMOKE' if SMOKE else 'FULL'} SUMMARY")
print("=" * 70)
ST = {c: json.load(open(OUT / c / "stats_b.json")) for c in COUNTRIES}
for c, s in ST.items():
    line = f"  {c:<8} queries {s['queries']:>9,}   new-layer pairs {s['pairs']:>11,} ({s['pairs_per_query']:.0f}/query)"
    if MODE == "train":
        line += (f"\n           recall: old {s['recall_old']*100:6.2f}%   new layers {s['recall_new']*100:6.2f}%"
                 f"   COMBINED {s['recall_combined']*100:6.2f}%   (held-out: old "
                 f"{s['recall_old_heldout']*100:.2f}% -> {s['recall_combined_heldout']*100:.2f}%)"
                 f"\n           layers alone: " + ", ".join(f"{k} {v*100:.1f}%" for k, v in s["recall_by_layer"].items())
                 + f"   pairs not in old: {s['new_pairs_not_in_old']:,}")
    print(line)
if SMOKE:
    fq, fp = full_q / max(used_q, 1), full_p / max(used_p, 1)
    factor = {"pool": fp, "query": fq, "both": fq * fp}
    print(f"\n  ROUGH RUNTIME PROJECTION (queries x{fq:.0f}, pool x{fp:.0f})")
    total = 0.0
    for stage, (sec, kind) in TIMES.items():
        full = sec * factor.get(kind, 1.0)
        total += full
        print(f"    {stage:<32}{sec:>8.1f}s  ->  {full/60:6.1f} min")
    print(f"    {'TOTAL (full run)':<32}{'':>9}      {total/60:6.1f} min")
    print("  Rough: the GPU search scales well below this on a real T4 batch.")
    print("  Smoke uses a pool slice, so recall numbers above are NOT meaningful.")
    print("\n  CODE CHECK PASSED. Set SMOKE = False, then Save Version -> Save & Run All.")
else:
    print(f"\n  Output: {OUT}  -> send it to the Stage 2 side (zip, or share this notebook's output)")
log("PHASE B COMPLETE")
mem("final")


PHASE B TEST FULL SUMMARY
  France   queries   259,452   new-layer pairs  25,996,319 (100/query)
  India    queries   809,986   new-layer pairs  80,775,837 (100/query)
  US       queries   663,106   new-layer pairs  65,322,744 (99/query)

  Output: /kaggle/working/ckpt_b_test  -> send it to the Stage 2 side (zip, or share this notebook's output)
  [ 55.4 min] PHASE B COMPLETE
    [MEM] final                        RAM=  8.2 GB  free= 23.8 GB


In [10]:
MAKE_ZIP = False     # True only in an INTERACTIVE session when you will download the result.
                     # Keep False in Save & Run All: the output folder is saved automatically,
                     # and a zip would double the output size and the save time.
if not SMOKE:
    # Remove the unpacked copies of the input zips so the saved output is only
    # ckpt_b_<mode>/ (they are re-created from the inputs whenever needed).
    shutil.rmtree(WORK / "_unzipped", ignore_errors=True)
    shutil.rmtree(CACHE, ignore_errors=True)
if MAKE_ZIP and not SMOKE:
    shutil.make_archive(str(WORK / f"ckpt_b_{MODE}"), "zip", str(WORK), f"ckpt_b_{MODE}")
    z = WORK / f"ckpt_b_{MODE}.zip"
    print(f"{z}  {z.stat().st_size/1e9:.2f} GB  -> download from the Output panel")
print("contents of /kaggle/working (this is what a committed version saves):")
for p in sorted(WORK.iterdir()):
    size = sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) if p.is_dir() else p.stat().st_size
    print(f"  {p.name:<28} {size/1e9:6.2f} GB")

contents of /kaggle/working (this is what a committed version saves):
  __notebook__.ipynb             0.00 GB
  ckpt_b_test                    2.41 GB
